# Buổi 4 — Tự hoàn thành notebook kiểm tra đạo văn

Hôm nay ta ghép từ câu chữ đến ma trận điểm tương đồng. Cuối buổi, bạn nên tự dựng lại được notebook kiểm tra đạo văn trong b2.

**Đích học:** IDF → TF-IDF → cosine → so mọi cặp → heatmap.

Chạy từ trên xuống. Các hàm hôm qua được nhắc lại để notebook này tự chạy độc lập. Trước cell TODO, hãy tự gõ trước rồi mới xem phần lời giải phía dưới.

## 1. Thư viện và ví dụ nhỏ

NumPy dùng cho vector/matrix; Matplotlib và Seaborn vẽ heatmap. Nếu thiếu thư viện trong Colab/Jupyter, bỏ dấu # ở dòng pip, chạy cell, rồi chạy lại import.

In [ ]:
# Nếu thiếu thư viện, bỏ dấu # và chạy dòng này một lần:
# %pip install numpy matplotlib seaborn

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

docs = [
    "mèo ăn cá mèo",
    "chó ăn cá",
    "mèo uống sữa"
]

## 2. Ôn ba hàm từ buổi 3

preprocess đổi câu thành list; create_corpus tạo thứ tự cột chung; compute_tf tạo vector tỷ lệ cho một tài liệu. Hãy giữ nguyên cùng một vocabulary khi tính tất cả hàng.

In [ ]:
def preprocess(text):
    return text.lower().split()

def create_corpus(docs):
    processed_docs = []
    vocabulary_set = set()

    for text in docs:
        words = preprocess(text)
        processed_docs.append(words)
        for word in words:
            vocabulary_set.add(word)

    vocabulary = sorted(vocabulary_set)
    return processed_docs, vocabulary

def compute_tf(word_list, vocabulary):
    word_count = {}
    for word in word_list:
        word_count[word] = word_count.get(word, 0) + 1

    total_words = len(word_list)
    tf_vector = []
    for word in vocabulary:
        count = word_count.get(word, 0)
        if total_words == 0:
            tf_vector.append(0)
        else:
            tf_vector.append(count / total_words)
    return np.array(tf_vector)

processed_docs, vocabulary = create_corpus(docs)
print("Số tài liệu:", len(processed_docs))
print("Vocabulary:", vocabulary)

## 3. Tính IDF

TF chỉ xem một câu; IDF xem cả tập. N là số tài liệu. df(word) là số tài liệu có từ đó. Từ lặp trong cùng một tài liệu không làm df tăng nhiều lần.

Công thức notebook gốc dùng làm trơn: **log((N + 1) / (df + 1)) + 1**. np.log là log tự nhiên.

In [ ]:
def compute_idf(processed_docs, vocabulary):
    num_docs = len(processed_docs)
    idf_values = []

    for word in vocabulary:
        document_frequency = 0
        for doc in processed_docs:
            if word in doc:
                document_frequency += 1

        idf = np.log(
            (num_docs + 1) / (document_frequency + 1)
        ) + 1
        idf_values.append(idf)

    return np.array(idf_values)

idf_vector = compute_idf(processed_docs, vocabulary)
for word, value in zip(vocabulary, idf_vector):
    print(word, round(value, 3))

Tự kiểm tra ý nghĩa:
- “ăn” và “cá” có ở hai tài liệu, nên df bằng 2.
- “sữa” có ở một tài liệu, nên df bằng 1 và IDF của nó lớn hơn.
- Mỗi từ tạo đúng một số IDF, vì thế độ dài idf_vector bằng số từ trong vocabulary.

### TỰ LÀM
Không nhìn cell trên, viết lại compute_idf. Dùng docs ở đầu bài và kiểm tra riêng df của “mèo”.

## 4. Từ TF sang ma trận TF-IDF

Ta tạo từng hàng TF theo cùng vocabulary. Sau đó nhân với IDF theo cột. Nếu N là số tài liệu và V là số từ, TF matrix có shape (N, V); IDF có shape (V,); kết quả vẫn là (N, V). NumPy nhân vector IDF với từng hàng.

In [ ]:
def compute_tf_idf(docs):
    processed_docs, vocabulary = create_corpus(docs)
    idf_vector = compute_idf(processed_docs, vocabulary)

    tf_matrix = []
    for doc in processed_docs:
        tf_vector = compute_tf(doc, vocabulary)
        tf_matrix.append(tf_vector)

    tf_matrix = np.array(tf_matrix)
    tf_idf_matrix = tf_matrix * idf_vector
    return tf_idf_matrix, vocabulary

tf_idf_matrix, vocabulary = compute_tf_idf(docs)
print("Shape:", tf_idf_matrix.shape)
print("Cột:", vocabulary)
print(tf_idf_matrix)

Nếu lỗi shape, hãy đếm: số hàng phải bằng số tài liệu; số cột phải bằng số từ duy nhất. Hàm trả hai thứ nên khi gọi nhận hai tên theo cùng thứ tự: ma trận trước, vocabulary sau.

## 5. Viết cosine similarity

Tích vô hướng nhân các vị trí tương ứng rồi cộng lại. Norm là độ dài vector. Cosine chia tích vô hướng cho hai độ dài để đo hướng, không để văn bản dài hơn tự động có điểm lớn hơn.

Nếu một vector toàn 0, độ dài bằng 0; phải kiểm tra trước khi chia.

In [ ]:
def cosine_similarity(vec_a, vec_b):
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot_product / (norm_a * norm_b)

print(cosine_similarity(np.array([1, 0]), np.array([1, 0])))
print(cosine_similarity(np.array([1, 0]), np.array([0, 1])))

Hai output cần thấy là 1.0 và 0.0. Nếu chúng khác, kiểm tra thứ tự phép nhân/chia và nhánh norm bằng 0.

### TỰ LÀM
Tự gõ lại hàm từ công thức. Sau đó thử cùng vector, vector vuông góc và vector toàn 0.

## 6. So sánh tất cả tài liệu

Tạo ma trận vuông N×N. Ô [i, j] là điểm cosine giữa tài liệu i và j. Đường chéo là mỗi tài liệu so với chính nó; hai nửa của ma trận phải đối xứng.

In [ ]:
def check_plagiarism_all(docs):
    tf_idf_matrix, vocabulary = compute_tf_idf(docs)
    num_docs = len(docs)
    similarity_matrix = np.zeros((num_docs, num_docs))

    for i in range(num_docs):
        for j in range(num_docs):
            similarity_matrix[i, j] = cosine_similarity(
                tf_idf_matrix[i], tf_idf_matrix[j]
            )

    return similarity_matrix

similarity_matrix = check_plagiarism_all(docs)
print(similarity_matrix.round(3))
print("Shape:", similarity_matrix.shape)

Đọc ma trận:
- Hàng 0/cột 1 là điểm giữa tài liệu 0 và 1.
- Hàng 1/cột 0 phải bằng hàng 0/cột 1.
- Các ô đường chéo thường bằng 1.
- Điểm cao chỉ báo từ ngữ giống nhau; không tự kết luận đạo văn.

## 7. Vẽ heatmap

Heatmap tô màu mỗi ô theo điểm. annot=True in số lên ô; fmt định dạng ba chữ số thập phân; xlabel/ylabel đặt nhãn trục.

In [ ]:
sns.heatmap(
    similarity_matrix,
    annot=True,
    fmt=".3f",
    cmap="YlOrRd"
)
plt.xlabel("Tài liệu")
plt.ylabel("Tài liệu")
plt.title("Cosine similarity giữa các tài liệu")
plt.show()

## 8. Bài kiểm tra notebook trắng

Đóng phần hướng dẫn hoặc mở một notebook mới. Từ đầu, hãy viết theo thứ tự:

1. Import NumPy; tạo docs.
2. preprocess → create_corpus → compute_tf.
3. compute_idf theo đúng công thức làm trơn.
4. compute_tf_idf và kiểm tra shape.
5. cosine_similarity có xử lý vector 0.
6. check_plagiarism_all và heatmap.
7. Thay docs bằng ba đoạn mới, chạy lại toàn bộ.

### Tự chấm
Bạn đạt khi tự viết được các bước, giải thích N, df, shape (N, V), và chỉ một ô bất kỳ trên heatmap về đúng cặp tài liệu. Điểm tương đồng không phải bằng chứng chắc chắn đạo văn; đây là công cụ rà soát.

## Lời giải tóm tắt
Các hàm hoàn chỉnh của buổi học đã được xây ở các cell phía trên. Dùng cell này như checklist, không chép cả bài: preprocess, create_corpus, compute_tf, compute_idf, compute_tf_idf, cosine_similarity, check_plagiarism_all, heatmap.

Ngày mai ta chuyển sang notebook thứ hai: dùng pandas và scikit-learn để truy vấn câu hỏi trên tập tài liệu.